# param-grad-access — worked example 2: per-parameter grad L2 norms

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-grad-access`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Reading `.grad` per named parameter lets you build a diagnostic mapping name to gradient L2 norm. Parameters with `.grad is None` are omitted entirely so a missing gradient never masquerades as a zero norm.

## Worked solution

We implement `grad_norms(model)` that walks `model.named_parameters()`, skips any with `p.grad is None`, and records `p.grad.norm(p=2).item()` under the parameter's stable dotted name. We build a tiny two-layer model, run a real forward and backward so PyTorch populates `.grad`, then call the function. Every parameter that participated gets an entry; none are None. We print the resulting dict of name to norm, which lets us spot a dead (zero-norm) layer at a glance.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)

def grad_norms(model):
    out = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        out[name] = p.grad.norm(p=2).item()
    return out

model = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 1))
x = t.randn(8, 4)
model(x).sum().backward()
for name, norm in grad_norms(model).items():
    print(name, round(norm, 4))